# Video to sprite (section 13.7)
Period search and sharp resampling on a simulated clip.

In [ ]:
import json, numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display
ROOT = Path.cwd().parent
def show(a, scale=1):
    im = Image.fromarray(a if a.dtype == np.uint8 else (np.clip(a, 0, 1) * 255).astype(np.uint8))
    if scale != 1: im = im.resize((im.width * scale, im.height * scale), Image.NEAREST)
    display(im)
def fixture(case):
    d = ROOT / 'tests' / 'golden' / case
    from spriteguru.pipeline.matte import ingest
    from spriteguru.spec import SpriteSpec
    t = json.loads((d / 'truth.json').read_text())
    return ingest(d / 'sheet.png'), SpriteSpec.model_validate(t['spec']), t

In [ ]:
import asyncio
from spriteguru.providers.synthetic import SyntheticProvider
from spriteguru.providers.base import ProviderRequest
from spriteguru.character import png_bytes
start = png_bytes(np.full((512, 512, 3), (0, 255, 0), np.uint8))
req = ProviderRequest(op='video', model='minimax/h3-max/image-to-video', prompt='VIDEO: A knight performs a walk cycle in place. (#00FF00)', params={'duration': 5}, images=[start, start])
clip = asyncio.run(SyntheticProvider().run(req)).candidates[0].data

In [ ]:
from spriteguru.pipeline.video import video_to_frames
frames, info = video_to_frames(clip, 8, key='#00FF00')
print(info)
from spriteguru.report import contact_sheet
display(contact_sheet(frames))